In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction


In [3]:
from pathlib import Path
import os
import sqlite3
import pandas as pd

BASE_DIR = Path("data")

def load_airline_data():
    print("[1] Loading train and test CSV files…")

    train_path = BASE_DIR / "train.csv"
    test_path = BASE_DIR / "test.csv"

    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError("train.csv or test.csv not found in data/")

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    df = pd.concat([train, test], ignore_index=True)
    print(f"Loaded {len(df)} total rows.")

    return df


def build_airline_3nf_sqlite(db_path="airline.db"):
    print("=== BUILDING AIRLINE PASSENGER SATISFACTION (3NF) ===")

    print("\n[STEP 1] Load data")
    df = load_airline_data()

    print("\n[STEP 2] Create surrogate passenger_id")
    df = df.reset_index(drop=True)
    df["passenger_id"] = df.index + 1

    print("\n[STEP 3] Passenger dimension")
    passenger_dim = (
        df[
            [
                "passenger_id",
                "Gender",
                "Customer Type",
                "Type of Travel",
                "Class",
            ]
        ]
        .drop_duplicates()
    )

    print("\n[STEP 4] Flight dimension")
    flight_dim = (
        df[
            [
                "Flight Distance",
                "Departure Delay in Minutes",
                "Arrival Delay in Minutes",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    flight_dim["flight_id"] = flight_dim.index + 1

    print("\n[STEP 5] Merge flight_id into main dataframe")
    df = df.merge(
        flight_dim,
        on=[
            "Flight Distance",
            "Departure Delay in Minutes",
            "Arrival Delay in Minutes",
        ],
        how="left",
    )

    print("\n[STEP 6] Service ratings fact table")
    ratings_cols = [
        "Inflight wifi service",
        "Departure/Arrival time convenient",
        "Ease of Online booking",
        "Gate location",
        "Food and drink",
        "Online boarding",
        "Seat comfort",
        "Inflight entertainment",
        "On-board service",
        "Leg room service",
        "Baggage handling",
        "Checkin service",
        "Inflight service",
        "Cleanliness",
        "satisfaction",
    ]

    fact_ratings = df[
        ["passenger_id", "flight_id"] + ratings_cols
    ]

    print("\n[STEP 7] Create SQLite database")
    if os.path.exists(db_path):
        print("Existing DB found. Removing…")
        os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    print("Creating tables…")
    cur.executescript(
        """
        DROP TABLE IF EXISTS service_ratings;
        DROP TABLE IF EXISTS flight;
        DROP TABLE IF EXISTS passenger;

        CREATE TABLE passenger (
            passenger_id INTEGER PRIMARY KEY,
            gender TEXT NOT NULL,
            customer_type TEXT NOT NULL,
            travel_type TEXT NOT NULL,
            class TEXT NOT NULL
        );

        CREATE TABLE flight (
            flight_id INTEGER PRIMARY KEY,
            flight_distance INTEGER NOT NULL,
            departure_delay INTEGER NOT NULL,
            arrival_delay INTEGER
        );

        CREATE TABLE service_ratings (
            passenger_id INTEGER,
            flight_id INTEGER,
            inflight_wifi INTEGER,
            time_convenient INTEGER,
            online_booking INTEGER,
            gate_location INTEGER,
            food_drink INTEGER,
            online_boarding INTEGER,
            seat_comfort INTEGER,
            inflight_entertainment INTEGER,
            onboard_service INTEGER,
            legroom INTEGER,
            baggage INTEGER,
            checkin INTEGER,
            inflight_service INTEGER,
            cleanliness INTEGER,
            satisfaction TEXT,
            PRIMARY KEY (passenger_id, flight_id),
            FOREIGN KEY (passenger_id) REFERENCES passenger(passenger_id),
            FOREIGN KEY (flight_id) REFERENCES flight(flight_id)
        );
        """
    )

    print("\n[STEP 8] Insert data")

    cur.executemany(
        """
        INSERT INTO passenger
        VALUES (?, ?, ?, ?, ?)
        """,
        list(passenger_dim.itertuples(index=False, name=None)),
    )

    cur.executemany(
        """
        INSERT INTO flight
        VALUES (?, ?, ?, ?)
        """,
        list(
            flight_dim[
                [
                    "flight_id",
                    "Flight Distance",
                    "Departure Delay in Minutes",
                    "Arrival Delay in Minutes",
                ]
            ].itertuples(index=False, name=None)
        ),
    )

    cur.executemany(
        """
        INSERT INTO service_ratings
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        list(fact_ratings.itertuples(index=False, name=None)),
    )

    conn.commit()
    conn.close()

    print("\n=== DONE! SQLite DB created:", db_path, "===")


build_airline_3nf_sqlite("./data/airline.db")

=== BUILDING AIRLINE PASSENGER SATISFACTION (3NF) ===

[STEP 1] Load data
[1] Loading train and test CSV files…
Loaded 129880 total rows.

[STEP 2] Create surrogate passenger_id

[STEP 3] Passenger dimension

[STEP 4] Flight dimension

[STEP 5] Merge flight_id into main dataframe

[STEP 6] Service ratings fact table

[STEP 7] Create SQLite database
Existing DB found. Removing…
Creating tables…

[STEP 8] Insert data

=== DONE! SQLite DB created: ./data/airline.db ===
